# Inference Lab: Pricing Under Uncertainty


**Run this notebook in Google Colab.** The `demand` module is compiled for Linux and Python 3.13 only, so it will not import on a Mac or Windows machine.

The goal of this lab is to use what you've learned in this class to find the optimal price for something. You can't know for sure what the underlying curve of revenue vs price looks like. All you know is what you would know in the real world, information you gain by sampling-- you can market test any price using the demand function below. When you set the price, the market test runs for a certain amount of time, and then reports responses-- a list of 1s (bought) and 0s (didn't buy) for that price point. The total number in the sample, $n$, is therefore a random variable, given by `len(responses)`. The number of consumers who buy the product is `responses.sum().` Every time you run a market test, it takes time (literally, it will take a while to run, so don't do this lab last minute!).

In [6]:
import numpy as np

In [4]:
!git clone -q https://github.com/smgroves/demand_lab.git

%cd /content/demand_lab

from demand import demand

/content/demand_lab


In [ ]:
responses = demand(price=30.0)

print(len(responses), responses.sum())

/content/demand_lab
187 78


In [ ]:
responses_25 = demand(price=25.0)
print(len(responses_25), responses_25.sum())

152 64


In [51]:
def bootstrap(response, price, R):
  bootstraps = np.empty(R)
  for i in range(R):
    sample = np.random.choice(response, size=len(response), replace=True)
    revenue = (price * sample.sum()) / len(sample)
    bootstraps[i] = revenue
  se = bootstraps.std(ddof=1)
  return bootstraps, se

def confidence_intervals(bootstraps):
  low = np.percentile(bootstraps, 2.5)
  high = np.percentile(bootstraps, 97.5)
  return low, high

In [ ]:
bootstrap_30, se_30 = bootstrap(responses, 30, 10000)
bootstrap_25, se_25 = bootstrap(responses_25, 25, 10000)

low_30, high_30 = confidence_intervals(bootstrap_30)
low_25, high_25 = confidence_intervals(bootstrap_25)

print(bootstrap_30.mean())
print(bootstrap_25.mean())

print(se_30)
print(se_25)

print(low_30, high_30)
print(low_25, high_25)

bootstrap_diff = bootstrap_25 - bootstrap_30
diff_low, diff_high = confidence_intervals(bootstrap_diff)
print(diff_low, diff_high)

12.52370053475936
10.525674342105262
1.0980148073210074
0.9973910164840292
10.427807486631016 14.598930481283423
8.552631578947368 12.5
-4.911166619757951 0.9330143540669873


## 1. Compare two prices

Run one market test with price = 30. Then test price = 25. Which price earns more revenue per potential customer?

1. For each price, estimate the proportion $\hat{p}$ of customers who buy, and the revenue per customer $\hat{r}$.
2. Estimate the standard error of each $\hat{r}$ by bootstrapping.
3. Compute a 95% confidence interval for each $\hat{r}$.
4. Compute a 95% confidence interval for the difference $\hat{r}_{25} - \hat{r}_{30}$.
   
5. Interpret the estimates, SEs, and intervals. Report which price earns more profit, and how confident you are.



**Price = 30** <br>
- 78/187 = 41.7% of customers buy the product
- 78 * 30 = 2340, 2340 / 187 = $12.51 per customer
- Standard Error = 1.098
- 95% CI: [10.428, 14.599]
<br>

**Price = 25** <br>
- 64/152 = 42.1% of customers buy the product
- 64 * 25 = 1600, 1600 / 152 = $10.53 per customer
- Standard Error = 0.997
- 95% CI: [8.553, 12.5]
<br>

**CI for difference**: [-4.911, 0.933]
<br><br>

**Report** <br>
By the metric of the one-sample estimate and mean of the bootstrap estimates, price=30 has a higher revenue per customer than price=25. By the metric of the confidence intervals, the CI for price=30 generally has higher values than the CI for price=25, indicating that higher values for revenue per customer are plausible for price=30. However, the confidence interval for the difference of revenue between the two prices contains 0. This suggests that there is not enough evidence to say that the different price points sufficiently affect customer decisions, and we cannot decide on a superior price point. The standard errors between the two prices are roughly the same, indicating similar levels of variability between the revenue per customer of the samples.

In [ ]:
responses_30_2 = demand(price=30.0)
responses_25_2 = demand(price=25.0)

combined_response_30 = np.concatenate((responses, responses_30_2))
combined_response_25 = np.concatenate((responses_25, responses_25_2))

In [ ]:
combined_30_bootstrap, combined_30_se = bootstrap(combined_response_30, 30, 10000)
combined_25_bootstrap, combined_25_se = bootstrap(combined_response_25, 25, 10000)

low_30_combined, high_30_combined = confidence_intervals(combined_30_bootstrap)
low_25_combined, high_25_combined = confidence_intervals(combined_25_bootstrap)

combined_difference = combined_25_bootstrap - combined_30_bootstrap
combined_diff_low, combined_diff_high = confidence_intervals(combined_difference)

print("bootstrap mean $30:", combined_30_bootstrap.mean())
print("bootstrap mean $25:", combined_25_bootstrap.mean())
print("$30 SE:", combined_30_se)
print("$25 SE:", combined_25_se)
print("$30 CI:", [low_30_combined, high_30_combined])
print("$25 CI:", [low_25_combined, high_25_combined])
print("Difference CI:", [combined_diff_low, combined_diff_high])

bootstrap mean $30: 12.523849693251533
bootstrap mean $25: 10.133029850746269
$30 SE: 0.8184963593802865
$25 SE: 0.6624538095580086
$30 CI: [np.float64(10.950920245398773), np.float64(14.079754601226995)]
$25 CI: [np.float64(8.880597014925373), np.float64(11.417910447761194)]
Difference CI: [np.float64(-4.480358941488875), np.float64(-0.341475139639228)]


## 2. The cost of sampling

7. Run a second market test at each price. Pool the two price = 30 samples (`np.concatenate`), and pool the two price = 25 samples.
8. Recompute the SEs, the intervals, and the interval for the difference. By roughly what factor did the SE shrink? Explain why, using the SE formula.
   
9.  **Was the extra sampling worth it?** Every market test takes time, and while you're testing you aren't selling at your best price, so each test costs the company revenue. Did the first round already settle the question (its difference interval excluded 0)? If so, the second round cost time and bought nothing but a narrower interval. If not, did the second round settle it?


**Recomputed Stats** <br>
New \$30 SE: 0.818 <br>
New \$25 SE: 0.662 <br>
New \$30 CI: [10.951, 14.08] <br>
New \$25 CI: [8.881, 11.418] <br>
New Difference CI: [-4.48, -0.341] <br>
<br>


**Report** <br>
The standard error for price=30 shrunk by a factor of 1.098/0.818 = 1.342, while the standard error for price=25 shrunk by 0.997/0.662 = 1.506. The formula for standard error uses the sample size (square root of the sample size) as the denominator, meaning that the standard error will shrink as the sample size grows. <br><br>The difference interval of the extra sampling no longer includes 0, meaning the exrtra sampling now answers the question that the original samples failed to. Because 0 is now excluded from the interval, we can say that there is sufficient evidence to say that revenue per customer at price=25 is lower than revenue per customer at price=30.

## 3. Finding the optimal price

Now find the price that maximizes revenue per customer. Search prices between about \$5 and \$100.

Every market test takes time, and time spent testing is revenue lost: while you test, you aren't selling at your best price. Your confidence intervals tell you whether another test is worth running. **Use as few market tests as you can (<25 would be good).** Your grade will be based on, in this order, whether you:
- accurately used the information you have by accurately interpreting confidence intervals
- found a good price, close to the optimal price
- and used fewer tests.

If this sounds fun to you, run with it! If you are still very confused about what to do, read the hints below.

<details><Summary>Hints</summary>
<b>Hint 1:</b> There are a few ways you could search:

- Brute force: test every price on a fine grid. This takes a market test at every price, including prices that are obviously bad. This would be a bad choice.
- Hill climbing: compare the CI for two prices, move toward the better one, and repeat. For example, start from the 25/30 comparison above; based on which price is better, choose another price \$10 away in that direction (15 if 25 was better; 40 if 30 was better). Shrink the step size as you get close to a good price. If you can't detect a difference between two prices, either shrink the step size or increase the sample size (i.e. run another market test at the same price).
- other strategies that I won't mention here.
  
<b>Hint 2:</b> You may want to try starting your tests with a few different prices across the range; there may be local maxima you will get stuck in if you just hill climb towards a single maximum.

<b>Hint 3:</b> You can make up your own strategy or try a mix of strategies. However, keep in mind that your grade will reflect how well you ran A/B testing and interpreting confidence intervals, more than how complex your strategy is. Similarly, a lucky guess with a brute force strategy would not be a good reproducible strategy.

<b>Hint 4:</b>
Turn your code from part 1 into a function, so you can easily input responses, run bootstrapping, calculate SE, and calculate confidence intervals. Then make another function where you give it two confidence intervals and make a decision based on them: sample more at the same test, or move in one direction with some step size.
   </details>

**Before you run any tests,** write down your plan: which prices you'll test first, how you'll decide where to go next, and what your intervals need to show before you stop. Then follow it, and note anywhere you changed course and why.

For any method, stop when your intervals show that another test is unlikely to change your answer by enough to matter.

**Deliverable:** A ~2-3 paragraph write up of the strategy you chose, your final recommended price, a 95% CI for $r$ at that price, the number of market tests you ran, and one or two sentences on how your intervals told you when to stop.

In [46]:
import random

def difference_CIs(boot_1, boot_2):
  difference = boot_1 - boot_2
  low = np.percentile(difference, 2.5)
  high = np.percentile(difference, 97.5)

  if low <= 0 <= high:
    print("0 is in the CI, not enough evidence")
  else:
    print("0 is not in the CI, statistically significant")

  print([low, high])

def test_price_range(start, end): # bootstrap => return CI, standard error, bootstrap mean revenue
  price = random.randint(start, end)
  result = demand(price=price)

  bootstraps = np.empty(10000)
  for i in range(10000):
    sample = np.random.choice(result, size=len(result), replace=True)
    revenue = (price * sample.sum()) / len(sample)
    bootstraps[i] = revenue
  se = bootstraps.std(ddof=1)
  bootstrap_mean = bootstraps.mean()

  ci = [np.percentile(bootstraps, 2.5), np.percentile(bootstraps, 97.5)]

  return price, ci, se, bootstrap_mean, bootstraps


In [47]:
# test 11-20, 30-40, 40-50, 50-60, 60-70, 70-80, 80-90, 90-100 (exclude first number of range)
# pick a random value in each interval to test
price_11, ci_11, se_11, mean_11, boot_11 = test_price_range(11, 20)
price_30, ci_30, se_30, mean_30, boot_30 = test_price_range(31, 40)
price_40, ci_40, se_40, mean_40, boot_40 = test_price_range(41, 50)
price_50, ci_50, se_50, mean_50, boot_50 = test_price_range(51, 60)
price_60, ci_60, se_60, mean_60, boot_60 = test_price_range(61, 70)
price_70, ci_70, se_70, mean_70, boot_70 = test_price_range(71, 80)
price_80, ci_80, se_80, mean_80, boot_80 = test_price_range(81, 90)
price_90, ci_90, se_90, mean_90, boot_90 = test_price_range(91, 100)

In [48]:
print("11-20 Stats:", price_11, ci_11, se_11, mean_11)
print("31-40 Stats:", price_30, ci_30, se_30, mean_30)
print("41-50 Stats:", price_40, ci_40, se_40, mean_40)
print("51-60 Stats:", price_50, ci_50, se_50, mean_50)
print("61-70 Stats:", price_60, ci_60, se_60, mean_60)
print("71-80 Stats:", price_70, ci_70, se_70, mean_70)
print("81-90 Stats:", price_80, ci_80, se_80, mean_80)
print("91-100 Stats:", price_90, ci_90, se_90, mean_90)

11-20 Stats: 13 [np.float64(10.11111111111111), np.float64(11.48989898989899)] 0.3495372454197893 10.832144949494948
31-40 Stats: 39 [np.float64(15.07361963190184), np.float64(21.05521472392638)] 1.5243312502962025 17.944737423312883
41-50 Stats: 43 [np.float64(13.938682432432438), np.float64(20.62837837837838)] 1.725798613693056 17.143402702702705
51-60 Stats: 54 [np.float64(12.134831460674157), np.float64(19.415730337078653)] 1.829114810260675 15.773794382022473
61-70 Stats: 66 [np.float64(14.320754716981131), np.float64(22.10377358490566)] 2.018132776949035 18.08829622641509
71-80 Stats: 78 [np.float64(18.241935483870968), np.float64(30.822580645161292)] 3.2647989451883515 24.518356451612902
81-90 Stats: 85 [np.float64(7.8061224489795915), np.float64(19.948979591836736)] 3.1912127048375902 13.924647959183671
91-100 Stats: 99 [np.float64(1.706896551724138), np.float64(9.387931034482758)] 2.03029987373152 5.143647413793103


In [55]:
# compare CIs
difference_CIs(boot_11, boot_30) # 31-40 > 11-20
difference_CIs(boot_30, boot_40) # not enough evidence
difference_CIs(boot_30, boot_50) # not enough evidence
difference_CIs(boot_40, boot_50) # not enough evidence
difference_CIs(boot_50, boot_60) # not enough evidence
difference_CIs(boot_60, boot_70) # not enough evidence
difference_CIs(boot_70, boot_50) # 71-80 > 51-60
difference_CIs(boot_70, boot_80) # 71-80 > 81-90
difference_CIs(boot_80, boot_90) # 81-90 > 91-100

0 is not in the CI, statistically significant
[np.float64(-10.198518931647765), np.float64(-4.043316601598809)]
0 is in the CI, not enough evidence
[np.float64(-3.8286353838501093), np.float64(5.280591941634885)]
0 is in the CI, not enough evidence
[np.float64(-2.551778451781899), np.float64(6.750274005652437)]
0 is in the CI, not enough evidence
[np.float64(-3.423170361372609), np.float64(6.369951412086245)]
0 is in the CI, not enough evidence
[np.float64(-7.534025863896545), np.float64(3.1795500317998693)]
0 is in the CI, not enough evidence
[np.float64(-14.011259890444311), np.float64(1.047169811320753)]
0 is not in the CI, statistically significant
[np.float64(1.5788329104748087), np.float64(15.957412105835449)]
0 is not in the CI, statistically significant
[np.float64(1.5905200789993401), np.float64(19.328176431863067)]
0 is not in the CI, statistically significant
[np.float64(1.020232230823364), np.float64(16.507389162561577)]


In [57]:
def confidence_intervals_2(bootstraps):
  low = np.percentile(bootstraps, 2.5)
  high = np.percentile(bootstraps, 97.5)
  return [low, high]

In [54]:
# Use the best range from previous code block, test 5 more times counting by 2 (example: 31, 33, 35, 37, 39)
bootstrap_71, se_71 = bootstrap(demand(price=71), 71, 10000)
bootstrap_73, se_73 = bootstrap(demand(price=73), 73, 10000)
bootstrap_75, se_75 = bootstrap(demand(price=75), 75, 10000)
bootstrap_77, se_77 = bootstrap(demand(price=77), 77, 10000)
bootstrap_79, se_79 = bootstrap(demand(price=79), 79, 10000)
print(bootstrap_71.mean())
print(bootstrap_73.mean())
print(bootstrap_75.mean())
print(bootstrap_77.mean())
print(bootstrap_79.mean())

19.072431052631583
19.13441693989071
20.839508670520228
24.151435
21.475182119205297


In [58]:
print(confidence_intervals_2(bootstrap_71))
print(confidence_intervals_2(bootstrap_73))
print(confidence_intervals_2(bootstrap_75))
print(confidence_intervals_2(bootstrap_77))
print(confidence_intervals_2(bootstrap_79))

[np.float64(14.573684210526316), np.float64(23.542105263157893)]
[np.float64(14.759562841530055), np.float64(23.934426229508198)]
[np.float64(16.040462427745666), np.float64(26.01156069364162)]
[np.float64(19.6), np.float64(29.05)]
[np.float64(15.695364238410596), np.float64(27.205298013245034)]


In [56]:
bootstrap_61, se_61 = bootstrap(demand(price=61), 61, 10000)
bootstrap_63, se_63 = bootstrap(demand(price=63), 63, 10000)
bootstrap_65, se_65 = bootstrap(demand(price=65), 65, 10000)
bootstrap_67, se_67 = bootstrap(demand(price=67), 67, 10000)
bootstrap_69, se_69 = bootstrap(demand(price=69), 69, 10000)
print(bootstrap_61.mean())
print(bootstrap_63.mean())
print(bootstrap_65.mean())
print(bootstrap_67.mean())
print(bootstrap_69.mean())

22.778137362637363
15.717479166666665
18.069228310502282
18.548520512820513
17.181840609137055


In [59]:
print(confidence_intervals_2(bootstrap_61))
print(confidence_intervals_2(bootstrap_63))
print(confidence_intervals_2(bootstrap_65))
print(confidence_intervals_2(bootstrap_67))
print(confidence_intervals_2(bootstrap_69))

[np.float64(18.434065934065934), np.float64(27.14835164835165)]
[np.float64(12.25), np.float64(19.25)]
[np.float64(14.246575342465754), np.float64(21.963470319634702)]
[np.float64(14.430769230769231), np.float64(22.676923076923078)]
[np.float64(13.309644670050762), np.float64(21.365482233502537)]


**Report** <br>
First, the possible range should be split into smaller ranges, and a price point should be chosen for each one. Here, we divide the range 5-100 into intervals of 10. We ignore the 20-30 interval as it was already tested with price points 25 and 30. Prices \$10 or below are excluded entirely. At these low price points, the best case scenario (when everyone in the sample makes a purchase) still does not beat the revenue per customer given by price points \$25 and \$30, so there is no chance that those low price points will be strong contenders for highest revenue per customer. After this, we take the range with the highest bootstrap mean and do a difference CI test with ranges with similar bootstrap means. The range with the highest bootstrap mean will be split further into 5 equally spaced values within that range, and each value's bootstrap mean will be tested, for a total of 13 market tests. When a range's CI contains 0 compared to at most one other range, then we stop and use those one or two ranges to test further.
<br><br>
The range with the highest bootstrap mean and the highest plausible values in the confidence interval is the range of 71-80. Specifically, 78 was the randomly chosen value in this range. Testing with this price gave a bootstrap mean of \$24.52, a confidence interval of [\$18.24, \$30.82], and a standard error of 3.26. Though the price gave the highest bootstrap mean and CI, it also gave the highest standard error, suggesting high variability between samples compared to the other prices. In addition, comparing this interval to the 61-70 range yielded a 95% CI containing 0, which suggests that the evidence isn't strong enough to prove a difference between the 71-80 and 61-70 range. Because of this, we increase the number of tests to 18 tests, 5 additional tests for the 61-70 range. We test 71,73,75,77,79, and then 61,63,65,67,69.
<br><br>
Among the CIs of these 10 price points, there is overlap between all of them with the exception of the CIs at price=77 and price=63. For the CIs of the prices between 71-80, there is significant overlap between them, suggesting a high likelihood of 0 being in the difference CI. The CIs of the 71-80 values generally contain higher plausible values than the CIs of the 61-70 values, though again there is significant overlap. Given the results, the final recommended price is $78, which has the highest bootstrap mean at \$24.52 and the highest 95% CI at [\$18.24, \$30.82].